In [ ]:
import json, pickle, math
import numpy as np
import torch
import torch.nn.functional as F
from model_arch import *

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PAD_ID, EOS_ID = 0, 3

In [ ]:
test = pickle.load(open("tok_test.pkl", "rb"))

model = DecoderLM(LMConfig(vocab_size=50260))
model.load_state_dict(torch.load("dec.pt", map_location="cpu"))
model.to(DEVICE)
model.eval()

In [ ]:
def make_chunks(doc, max_chunks=None):
    n = math.ceil(len(doc) / 1023)
    chunks = []
    for i in range(n):
        piece = doc[i * 1023:(i + 1) * 1023].astype(np.int64)
        chunks.append(np.append(piece, EOS_ID))
    return chunks

def chunk_vectors(model, chunks):
    lengths = torch.tensor([len(c) for c in chunks], device=DEVICE)
    T = int(lengths.max())
    ids = torch.full((len(chunks), T), PAD_ID, dtype=torch.long)
    for i, c in enumerate(chunks):
        ids[i, :len(c)] = torch.from_numpy(c)
    ids = ids.to(DEVICE)

    with torch.autocast("cuda", dtype=torch.float16):
        x = model.tok_emb(ids) + model.pos_emb[:T]
        for block in model.blocks:
            x = block(x)
        x = model.ln_f(x)

    last = x[torch.arange(len(chunks), device=DEVICE), lengths - 1]
    return F.normalize(last.float(), dim=-1)

@torch.no_grad()
def doc_vectors(model, docs, max_chunks=None, chunk_batch=64):
    all_chunks, owner = [], []
    for d_i, doc in enumerate(docs):
        cs = make_chunks(doc, max_chunks)
        all_chunks += cs
        owner += [d_i] * len(cs)
    
    owner = torch.tensor(owner, device=DEVICE)
    sums = torch.zeros(len(docs), model.cfg.d_model, device=DEVICE)
    
    for s in range(0, len(all_chunks), chunk_batch):
        v = chunk_vectors(model, all_chunks[s:s + chunk_batch])
        sums = sums.index_add(0, owner[s:s + chunk_batch], v)
        
    return F.normalize(sums, dim=-1)

In [ ]:
cand_ids = list(test["cands"].keys())
cand_vecs = doc_vectors(model, [test["cands"][c] for c in cand_ids])

q_ids = list(test["queries"].keys())
q_vecs = doc_vectors(model, [test["queries"][q] for q in q_ids])

sims = q_vecs @ cand_vecs.T
col = {c: i for i, c in enumerate(cand_ids)}

for r, q in enumerate(q_ids):
    if q in col:
        sims[r, col[q]] = float("-inf")
        
top7 = sims.topk(7, dim=1).indices.cpu().numpy()

prediction = {}
for r, q in enumerate(q_ids):
    prediction[q] = [cand_ids[j] for j in top7[r]]
    
with open("prediction.json", "w") as f:
    json.dump(prediction, f, indent=2)